# Эксперемент с сырыми данными

In [4]:
# Базовые библиотеки
import pandas as pd 
from tqdm.notebook import tqdm

# Функции обучения моделей
from models.catboost import test_catboost
from models.xgboost import test_xgb_manual_encoding, test_xgb_with_recoder

# Вспомогательные утилиты
from models.utils import (
    calculation_metrics, 
    calculation_info_data, 
    load_data, 
    count_unique_data
)

## Конфигурации

Для сравнительнения XGBoost и CatBoost создадим сбалансированные параметры, которые позволят честно сравнить эффектиность архитектур. Сфокусируемся на сравнении встроенных механизмов обработки категориальных признаков. Таким образом мы выявим сильные стороны каждой реализации в работе с сырыми данными. 

In [5]:
xgb_params = {
    'n_estimators': 100,        # Количество деревьев    
    'max_depth': 6,             # Максимальная глубина деревьев
    'learning_rate': 0.1,       # Скорость обучения
    'subsample': 0.8,           # Доля данных для каждого дерева
    'colsample_bytree': 0.8,    # Доля признаков для каждого дерева
    'reg_lambda': 1.0,          # L2 регуляризация
    'random_state': 42,         # Seed для воспроизводимости
    'n_jobs': -1,               # Использование всех ядер
    
    # Специфичные для категорий:
    'tree_method': 'hist',      # Метод для построени
    'max_cat_to_onehot': 5      # Порог для one-hot кодирования
}

catboost_params = {
    'iterations': 100,          # Количество итераций (аналог n_estimators)
    'depth': 6,                 # Максимальная глубина деревьев
    'learning_rate': 0.1,       # Скорость обучения
    'subsample': 0.8,           # Доля данных для каждого дерева
    'colsample_bylevel': 0.8,   # Доля признаков для каждого дерева
    'l2_leaf_reg': 1.0,         # L2 регуляризация
    'random_state': 42,         # Seed для воспроизводимости
    'thread_count': -1,         # Использование всех ядер

    # Специфичные для категорий
    'one_hot_max_size': 5,      # Порог one-hot кодирования категорий
    
    # Выкл отображения в консоли
    'verbose': False            # Отключение вывода обучения
}

Так же для эксперемента создадим конфигурационный список `testing_models`. Такой подход обеспечит модульность и расширяемость(если понадобится мы с лёгкостью добавим или изменим пайплайн обучения). Так же обратите внимание, что мы создали 2 варианта обучения XGBoost. В первом варианте мы обучаем его с предобработкой категориальных данных с использованием двух кодировщиков `OneHotEncoder` и `OrdinalEncoder` из бибилиотеки `sklearn`. Во втром варианте вся работа с кодировкой ложиться на ре-кодер, который мы с вами разбирали ранее.

In [6]:
testing_models = [
    {
        "name": "catBoost",
        "model_func": test_catboost,
        "params": catboost_params,
    },
    {
        "name": "xgb_manual_encoder",
        "model_func": test_xgb_manual_encoding,
        "params": xgb_params
    },
    {
        "name": "xgb_with_recoder",
        "model_func": test_xgb_with_recoder,
        "params": xgb_params
    },
]

## Датасеты который мы берём для эксперемента

### [House Prices - Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques)
Предсказываем цену домов

In [7]:
info_datasets = []
dataset = {
    "name": "House Prices - Advanced Regression Techniques",
    "url": "https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/overview",
    "file": "data\\house_prices.csv",
    "task": "reg",
    "target": "SalePrice",
    "trash_columns": ['Id']
}
info_datasets.append(dataset)

### [Mental Health Dataset](https://www.kaggle.com/datasets/alamshihab075/mental-health-dataset)
Предсказываем пол человека в зависимости от депрессии

In [8]:
dataset = {
    "name": "Mental Health Dataset",
    "url": "https://www.kaggle.com/datasets/alamshihab075/mental-health-dataset",
    "file": "data\\mental_health_dataset.csv",
    "task": "clf",
    "target": "Gender",
    "trash_columns": []
}
info_datasets.append(dataset)

### [Life Style Data](https://www.kaggle.com/datasets/jockeroika/life-style-data) 

Предсказываем пол в завивисмости от жизненной активности

In [9]:
dataset = {
    "name": "Life Style Data",
    "url": "https://www.kaggle.com/datasets/jockeroika/life-style-data",
    "file": "data\\life_style_data.csv",
    "task": "clf",
    "target": "Gender",
    "trash_columns": []
}
info_datasets.append(dataset)

### [All Computer Prices](https://www.kaggle.com/datasets/paperxd/all-computer-prices)
Предсказываем цену компьютера

In [10]:
dataset = {
    "name": "All Computer Prices",
    "url": "https://www.kaggle.com/datasets/paperxd/all-computer-prices",
    "file": "data\\computer_prices_all.csv",
    "task": "reg",
    "target": "price",
    "trash_columns": []
}
info_datasets.append(dataset)

### [Flight Delay Dataset — 2024](https://www.kaggle.com/datasets/hrishitpatil/flight-data-2024)
Предсказываем количество минут, на которое задержался рейс

In [11]:
dataset = {
    "name": "Flight Delay Dataset — 2024",
    "url": "https://www.kaggle.com/datasets/hrishitpatil/flight-data-2024",
    "file": "data\\flight_data_2024.csv",
    "task": "reg",
    "target": "late_aircraft_delay",
    "trash_columns": ['year']
}
info_datasets.append(dataset)

### [Predicting Road Accident Risk](https://www.kaggle.com/competitions/playground-series-s5e10) 

Предсказываем вероятность возникноваения ДТП

In [12]:
dataset = {
    "name": "Predicting Road Accident Risk",
    "url": "https://www.kaggle.com/competitions/playground-series-s5e10",
    "file": "data\\road_accident_risk.csv",
    "task": "reg",
    "target": "accident_risk",
    "trash_columns": ['id']
}
info_datasets.append(dataset)

### [Binary Classification with a Bank Dataset](https://www.kaggle.com/competitions/playground-series-s5e8) 

Предсказываем подпишет ли клиент срочный банковский депозит.

In [13]:
dataset = {
    "name": "Binary Classification with a Bank Dataset",
    "url": "https://www.kaggle.com/competitions/playground-series-s5e8",
    "file": "data\\classification_bank_dataset.csv",
    "task": "clf",
    "target": "y",
    "trash_columns": ['id']
}
info_datasets.append(dataset)

## Обучение

Теперь запускаем паплайн для сравнения моделей.

In [ ]:
results = []

pbar_datasets = tqdm(
    info_datasets,
    unit="dataset",
    colour="#037503",
    leave=True
)

for info_dataset in pbar_datasets:
    pbar_datasets.set_description(f"Dataset '{info_dataset['name']}'")

    X_train, X_test, y_train, y_test = load_data(info_dataset)
    info_columns_dataset = calculation_info_data(X_train)

    pbar_models = tqdm(
        testing_models,
        unit="model",
        colour="#09ff00",
        leave=False
    )

    for test_model in pbar_models:
        pbar_models.set_description(f"Processing '{test_model['name']}'")
        # Вызываем функцию модели
        model, y_pred, train_time = test_model["model_func"](
            X_train.copy(), 
            X_test.copy(), 
            y_train.copy(), 
            params=test_model["params"],
            task_type=info_dataset["task"]
        )
        
        metrics = calculation_metrics(y_pred, y_test, info_dataset["task"])
        results.append({
            "model_name": test_model["name"],
            "dataset_name": info_dataset["name"],
            "train_time": train_time,
            **metrics,
            **info_columns_dataset,
        })
    pbar_models.close()
pbar_datasets.close()

  0%|          | 0/7 [00:00<?, ?dataset/s]

  0%|          | 0/3 [00:00<?, ?model/s]

  0%|          | 0/3 [00:00<?, ?model/s]

  0%|          | 0/3 [00:00<?, ?model/s]

KeyboardInterrupt: 

От общего к частному - после масштабного эксперимента с 7 датасетами(результат которых мы разберём позже) погрузимся в детальный разбор задачи прогнозирования арестов по данным о преступлениях. Давайте разберём датасет [Crime & Consequence: A Metropolitan Dataset](https://www.kaggle.com/datasets/har5hdeep5harma/chicago-crime-incidents-2001-to-present).

In [12]:
info_dataset = {
    "name": "Crime & Consequence: A Metropolitan Dataset",
    "url": "https://www.kaggle.com/datasets/har5hdeep5harma/chicago-crime-incidents-2001-to-present",
    "file": "data\\Chicago_Crimes_2001_to_Present.csv",
    "task": "clf",
    "target": "Arrest",
    "trash_columns": ['ID', 'Case Number']
}

Давайте посмтрим количество уникальных значений для каждой фичи

In [13]:
X_train, X_test, y_train, y_test = load_data(info_dataset)
info_columns_dataset = calculation_info_data(X_train)

unique_counts = count_unique_data(X_train)

In [14]:
pd.DataFrame(
    data={
        'columns': list(key for key in unique_counts.keys()),
        'n_unique': list(val['n_unique'] for val in unique_counts.values()),
        'dtype': list(val['dtype'] for val in unique_counts.values())
    }
).sort_values('n_unique', ascending=False, ignore_index=True)

,columns,n_unique,dtype
0,Date,3022335,object
1,Location,839274,object
2,Latitude,838172,float64
3,Longitude,837646,float64
4,Y Coordinate,129517,float64
5,X Coordinate,78408,float64
6,Block,63540,object
7,Updated On,7145,object
8,Description,562,object
9,IUCR,415,object


Вы думаете, XGBoost справится с такой нагрузкой?

In [15]:
results_beta = []

pbar_models = tqdm(
    testing_models[:2],
    unit="model",
    colour="#09ff00",
    leave=False
)

for test_model in pbar_models:
    pbar_models.set_description(f"Processing '{test_model['name']}'")
    # Вызываем функцию модели
    model, y_pred, train_time = test_model["model_func"](
        X_train.copy(), 
        X_test.copy(), 
        y_train.copy(), 
        params=test_model["params"],
        task_type=info_dataset["task"]
    )
    
    metrics = calculation_metrics(y_pred, y_test, info_dataset["task"])
    results_beta.append({
        "dataset_name": info_dataset["name"],
        "model_name": test_model["name"],
        "train_time": train_time,
        **metrics,
        **info_columns_dataset,
    })
pbar_models.close()

  0%|          | 0/2 [00:00<?, ?model/s]

In [ ]:
%%time
# Модель с рекодером у нас 2-рая в с писке
# Включаем детальный прогресс только для отладки
debug_params = testing_models[2]["params"].copy()
debug_params.update({
    'verbose': 1,
    'verbose_eval': 5, 
})
test_model = testing_models[2]

model, y_pred, train_time = test_model["model_func"](
    X_train.copy(), 
    X_test.copy(), 
    y_train.copy(), 
    params=debug_params,
    task_type=info_dataset["task"]
)

metrics = calculation_metrics(y_pred, y_test, info_dataset["task"])
results_beta.append({
    "dataset_name": info_dataset["name"],
    "model_name": test_model["name"],
    "train_time": train_time,
    **metrics,
    **info_columns_dataset,
})S

XGBoost с ре-кодером обучался больше часа и не смог обучиться на данных. В то время как Catboost и XGBoost с ручным кодированием обучились менее чем за 6 минут в общей сумме.

In [16]:
pd.DataFrame(results_beta)

,dataset_name,model_name,train_time,accuracy,f1,recall,precision,data_count_columns,data_count_row,data_count_num_columns,data_count_cat_columns,data_portion_num_columns,data_portion_cat_columns
0,Crime & Consequence: A Metropolitan Dataset,catBoost,179.558494,0.8924,0.7493,0.6367,0.9104,19,6737676,9,10,0.474,0.526
1,Crime & Consequence: A Metropolitan Dataset,xgb_manual_encoder,135.703350,0.8910,0.7455,0.6321,0.9084,19,6737676,9,10,0.474,0.526


**Итог:** в большом датасете *Crime & Consequence: A Metropolitan Dataset* (6.7M записей, 53% категориальных признаков) с высоким количеством уникальных значений в категориальных признаках CatBoost демонстрирует не только стабильность, но и практическое превосходство. Он не только обучается успешно, но и показывает лучшие результаты, в то время как XGBoost с автоматическим кодированием оказывается неприменимым.


## Разбор результатов

Настало время подвести итоги нашего эксперимента. Мы протестировали модели на множестве датасетов - теперь давайте рассмотри полученные результаты и проведём сравнительный анализ производительности алгоритмов и качества предсказаний в задачах классификации и регрессии.

In [ ]:
df_clf_task = pd.DataFrame(results).drop(['r2', 'rmse', 'mae'], axis=1).dropna(ignore_index=True)
df_reg_task = pd.DataFrame(results).drop(['accuracy', 'f1', 'recall', 'precision'], axis=1).dropna(ignore_index=True)

### Задачи с классификацией

In [ ]:
df_clf_task

,model_name,dataset_name,train_time,data_count_columns,data_count_row,data_count_num_columns,data_count_cat_columns,data_portion_num_columns,data_portion_cat_columns,accuracy,f1,recall,precision
0,catBoost,Mental Health Dataset,6.036316,16,209062,0,16,0.000,1.000,0.9256,0.9596,0.9996,0.9227
1,xgb_manual_encoder,Mental Health Dataset,2.911505,16,209062,0,16,0.000,1.000,0.9618,0.9789,0.9999,0.9586
2,xgb_with_recoder,Mental Health Dataset,2.773760,16,209062,0,16,0.000,1.000,0.9551,0.9753,0.9999,0.9518
3,catBoost,Life Style Data,3.977352,53,16000,39,14,0.736,0.264,0.4868,0.4957,0.5148,0.4780
4,xgb_manual_encoder,Life Style Data,0.726083,53,16000,39,14,0.736,0.264,0.4985,0.4980,0.5077,0.4887
5,xgb_with_recoder,Life Style Data,1.049625,53,16000,39,14,0.736,0.264,0.4963,0.4954,0.5046,0.4865
6,catBoost,Binary Classification with a Bank Dataset,11.738585,16,600000,7,9,0.438,0.562,0.9275,0.6690,0.6040,0.7496
7,xgb_manual_encoder,Binary Classification with a Bank Dataset,5.237780,16,600000,7,9,0.438,0.562,0.9318,0.6961,0.6434,0.7580
8,xgb_with_recoder,Binary Classification with a Bank Dataset,4.793217,16,600000,7,9,0.438,0.562,0.9326,0.7007,0.6502,0.7599


**Mental Health Dataset** (100% категориальные признаки):
- XGBoost с ручным кодированием показывает наилучший результат: 96.18% accuracy
- CatBoost отстаёт по точности, но демонстрирует сбалансированные метрики
- Обе версии XGBoost обучаются в 2 раза быстрее CatBoost

**Life Style Data** (73% числовых признаков):
- Все модели показывают скромные результаты (около 50% accuracy)
- XGBoost с ручным кодированием незначительно лидирует
- Время обучения CatBoost в 5 раз больше при сравнимом качестве

**Bank Dataset** (сбалансированные типы признаков):
- XGBoost с автоматическим кодированием демонстрирует лучший F1-score: 70.07%
- Все модели показывают высокую точность (>92%), но разный баланс precision/recall
- XGBoost варианты обучаются в 2 раза быстрее CatBoost

XGBoost выигрывает в скорости и точности, CatBoost в стабильности метрик. Первый оптимален для экспериментов и быстрого прототипирования, второй для промышленной эксплуатации, где важна надежность работы модели на разнообразных данных, особенно с преобладанием категориальных признаков.

### Задачи с регрессией

In [ ]:
df_reg_task

,model_name,dataset_name,train_time,data_count_columns,data_count_row,data_count_num_columns,data_count_cat_columns,data_portion_num_columns,data_portion_cat_columns,r2,rmse,mae
0,catBoost,House Prices - Advanced Regression Techniques,2.428524,79,1168,36,43,0.456,0.544,0.8654,28658.26,17025.43
1,xgb_manual_encoder,House Prices - Advanced Regression Techniques,0.465448,79,1168,36,43,0.456,0.544,0.8984,25020.42,15807.90
2,xgb_with_recoder,House Prices - Advanced Regression Techniques,0.460096,79,1168,36,43,0.456,0.544,0.8916,25440.36,15584.49
3,catBoost,All Computer Prices,4.078966,32,80000,19,13,0.594,0.406,0.8615,197.99,138.67
4,xgb_manual_encoder,All Computer Prices,2.048132,32,80000,19,13,0.594,0.406,0.8531,205.03,143.60
5,xgb_with_recoder,All Computer Prices,34.141360,32,80000,19,13,0.594,0.406,0.8203,216.52,149.35
6,catBoost,Flight Delay Dataset — 2024,115.896997,33,5663264,24,9,0.727,0.273,0.9784,4.34,0.96
7,xgb_manual_encoder,Flight Delay Dataset — 2024,55.221895,33,5663264,24,9,0.727,0.273,0.9298,7.69,0.76
8,xgb_with_recoder,Flight Delay Dataset — 2024,43.152280,33,5663264,24,9,0.727,0.273,0.9144,8.42,0.98
9,catBoost,Predicting Road Accident Risk,4.057827,12,414203,4,8,0.333,0.667,0.8680,0.06,0.04


**House Prices - Advanced Regression Techniques** (сбалансированные типы признаков):
* XGBoost с ручным кодированием показывает наилучший результат: R²=0.8984
* CatBoost отстаёт по точности (R² = 0.8654), но демонстрирует хорошую стабильность
* Обе версии XGBoost обучаются в 5.5 раз быстрее CatBoost

**All Computer Prices** (59% числовых признаков):
* CatBoost лидирует по точности: R² = 0.8615
* XGBoost с автоматическим кодированием показывает неожиданно низкий результат (R² = 0.8203) и долгое обучение... 
* XGBoost с ручным кодированием показывае конкурентное качество при быстром обучении

**Flight Delay Dataset — 2024** (5.6M записей, 73% числовых признаков):
* CatBoost демонстрирует исключительное качество: R² = 0.9784
* Обе версии XGBoost значительно отстают по точности (R² = 0.92)
* Разница в скорости обучения в 2 раз в пользу XGBoost

**Predicting Road Accident Risk** (67% категориальных признаков):
* Все модели показывают идентичные результаты (R² ≈ 0.87)
* XGBoost с автоматическим кодированием обучается быстрее всех
* Минимальная разница в метриках демонстрирует сходимость алгоритмов на некоторых типах данных

XGBoost сохраняет преимущество в скорости, но CatBoost демонстрирует превосходство на больших и сложных датасетах, особенно там, где требуется максимальная точность предсказаний. Первый остаётся оптимальным для быстрого прототипирования и экспериментов, второй для систем с высокими требованиями к качеству, не требующих быстрого обучения. Особенно показателен случай с *Flight Delay Dataset*, где CatBoost показал исключительное качество на 5.6 миллионах записей.

# ИТОГ эксперемента:

...